<a href="https://colab.research.google.com/github/Hrishik1033/FedAVG-Implementation/blob/main/FedAVG(Tensorflow).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading the Dataset

In [ ]:
import tensorflow as tf
import numpy as np

In [ ]:
def load_mnist():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

    # Bringing in the range of 0 to 1
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    x_train = x_train.reshape(-1, 28, 28, 1)
    x_test = x_test.reshape(-1, 28, 28, 1)

    return (x_train, y_train.astype("int64")), (x_test, y_test.astype("int64"))

# Partitioning the IID

In [ ]:
def partition_iid(y, num_clients):
    """Shuffle everything, then split into equal, random chunks.
    Every client's local distribution looks like the global distribution"""

    idx = np.random.permutation(len(y)) # Returns in shuffled order
    return np.array_split(idx, num_clients) #Equally split the data into num_clients


# Building the model

In [ ]:
from keras.layers import Input
from keras import layers
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.models import Model
def build_model():
  input = Input(shape=(28, 28, 1))
  x = Conv2D(filters=32, kernel_size=(3, 3), activation='relu')(input)
  x = MaxPooling2D((2, 2))(x)

  x = Conv2D(64, (3, 3), activation='relu')(x)
  x = MaxPooling2D((2, 2))(x)

  x = Conv2D(128, (3, 3), activation='relu')(x)

  x = Flatten()(x)
  x = Dense(512, activation='relu')(x)
  output = Dense(10, activation='softmax')(x)
  model = Model(inputs=input, outputs=output)
  return model

In [ ]:
def compile_model(model,learning_rate):
  model.compile(
      loss=tf.keras.losses.SparseCategoricalCrossentropy(),
      optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
      metrics=["accuracy"]
  )
  return model

# Get the weights of the local models from the clients

In [ ]:
def get_weights(model):
  weights = [w for w in model.get_weights()]
  return weights


# Setting the weights

In [ ]:
def set_weights(model, weights):
  model.set_weights(weights)


# Client Update

In [ ]:
def client_update(global_weights, x_k, y_k, E, B, lr,debug = False):
    local_model = compile_model(build_model(), lr)
    set_weights(local_model, global_weights)
    batch_size = len(x_k) if B is None else B   # B=None means "full batch", i.e. B=infinity in the paper

    hist = local_model.fit(x_k, y_k, epochs=E, batch_size=batch_size, verbose=1)
    if debug:
        final_loss = hist.history["loss"][-1]
        final_acc = hist.history["accuracy"][-1]
        print(f"      client n={len(x_k):4d} | local final loss {final_loss:.3f} "
              f"| local final acc {final_acc:.3f}")
    return get_weights(local_model), len(x_k)

# Doing the FedAVG Algorithm

#Outer loop pass 1 (client A, n_k=600):
#    inner loop i=0: avg[0] += (600/total) * A's conv1_kernel
#    inner loop i=1: avg[1] += (600/total) * A's conv1_bias
 #   inner loop i=2: avg[2] += (600/total) * A's conv2_kernel
#    ... (continues through all 6 layers)

#Outer loop pass 2 (client B, n_k=598):
#    inner loop i=0: avg[0] += (598/total) * B's conv1_kernel   <- #ADDS onto what A already contributed
#    inner loop i=1: avg[1] += (598/total) * B's conv1_bias
#    ...

#Outer loop pass 3 (client C, n_k=603):
#    inner loop i=0: avg[0] += (603/total) * C's conv1_kernel   <- #ADDS onto A + B's contribution
#    ...

In [ ]:
def federated_average(client_weights_list, client_sizes):
    total = sum(client_sizes)
    avg = [np.zeros_like(w) for w in client_weights_list[0]] # Initializing with zeros first as no data is there
    for weights, n_k in zip(client_weights_list, client_sizes):
        for i, w in enumerate(weights):
            avg[i] += (n_k / total) * w
    return avg

# Servers main algorithm

In [ ]:
def run_fedavg(x_train, y_train, x_test, y_test, client_indices,
               num_rounds=60, C=0.2, E=5, B=10, lr=0.01, log_every=1):
    K = len(client_indices)
    m = max(int(C * K), 1)   # number of clients sampled each round

    global_model = compile_model(build_model(), lr)
    global_weights = get_weights(global_model)

    history = []
    for t in range(1, num_rounds + 1):
        #server selects a random fraction C of clients
        selected = np.random.choice(K, m, replace=False) # The replace  = False is there so that same client is not selected twice in same round

        client_weights_list, client_sizes = [], []
        for k in selected:
            idxs = client_indices[k]
            x_k, y_k = x_train[idxs], y_train[idxs]
            # each selected client trains locally and reports back
            w_k, n_k = client_update(global_weights, x_k, y_k, E, B, lr)
            client_weights_list.append(w_k)
            client_sizes.append(n_k)

        # server aggregates: weighted average of client models
        global_weights = federated_average(client_weights_list, client_sizes)
        set_weights(global_model, global_weights)

        if t % log_every == 0 or t == num_rounds:
            loss, acc = global_model.evaluate(x_test, y_test, verbose=1)
            print(f"Round {t:3d}/{num_rounds} | clients used: {m:3d} "
                  f"| test loss: {loss:.4f} | test acc: {acc:.4f}")
            history.append((t, loss, acc))

    return global_model, history

In [11]:
if __name__ == "__main__":
    print("Loading MNIST...")
    (x_train, y_train), (x_test, y_test) = load_mnist()

    NUM_CLIENTS = 100

    print("\n--- FedAvg on IID data ---")
    iid_clients = partition_iid(y_train, NUM_CLIENTS)
    run_fedavg(x_train, y_train, x_test, y_test, iid_clients)

    print("\n--- Baseline: FedSGD (E=1, B=infinity) for comparison ---")
    run_fedavg(x_train, y_train, x_test, y_test, iid_clients,
               num_rounds=15, C=0.1, E=1, B=None, lr=0.1)


Loading MNIST...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- FedAvg on IID data ---
Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.2350 - loss: 2.2845
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4417 - loss: 2.2211
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5183 - loss: 2.0669
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6283 - loss: 1.5781
Epoch 5/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7117 - loss: 0.9557
Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2317 - loss: 2.2777
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4383 - loss: 2.2003
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5083 - loss: 1.9837
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6100 - loss: 1.3935
Epoch 5/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7317 - loss: 0.8878
Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.2633 - loss

KeyboardInterrupt: 